In [ ]:
# import requirements
import os
import csv
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DATASET_ROOT = r"D:\lakehead\Deep Learning\project\full_house\train_test_val"
OUTPUT_CSV = "landmarks_dataset_handedness_fixed.csv"
# Path to the Pre-trained model (MediaPipe Hand Landmarker .task model file)
MODEL_PATH = r"D:\lakehead\Deep Learning\project\full_house\hand_landmarker.task"

#list of all english letters
CLASSES = [chr(i) for i in range(ord("A"), ord("Z") + 1)]
SPLITS = ["train", "val", "test"]

# Tell MediaPipe where the hand landmarker model file is located
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

# Configure the hand landmarker
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

#create the CSV to save the hand coordinates x,y,and z
header = []
for i in range(21):
    header.extend([f"x{i}", f"y{i}", f"z{i}"])
header += ["label", "split", "filepath", "handedness"]

success_count = 0
fail_count = 0

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)

    # Create the MediaPipe Hand Landmarker using the options above
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        # Loop through each split: train, val, test
        for split in SPLITS:
            for label in CLASSES:
                folder = os.path.join(DATASET_ROOT, split, label)

                for filename in os.listdir(folder):
                    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
                        continue

                    path = os.path.join(folder, filename)
                    image = cv2.imread(path)

                    if image is None:
                        fail_count += 1
                        continue

                    # Convert image from BGR (OpenCV default) to RGB (MediaPipe expects RGB)
                    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    # Convert the NumPy image into a MediaPipe Image object
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                    result = landmarker.detect(mp_image)
                    
                    # Check if at least one hand was detected
                    if result.hand_landmarks and len(result.hand_landmarks) > 0:
                        hand_landmarks = result.hand_landmarks[0]

                        handed_label = "Unknown"
                        if result.handedness and len(result.handedness) > 0:
                            handed_label = result.handedness[0][0].category_name

                        coords = []
                        for lm in hand_landmarks:
                            coords.append([lm.x, lm.y, lm.z])

                        coords = np.array(coords, dtype=np.float32)

                        # If the detected hand is the right hand,
                        # flip the x coordinates so that right hands look like left hands.
                        # This helps make the classifier invariant to left/right hand use.
                        if handed_label == "Right":
                            coords[:, 0] = 1.0 - coords[:, 0]

                        row = coords.flatten().tolist()
                        row.extend([label, split, path, handed_label])

                        writer.writerow(row)
                        success_count += 1
                    else:
                        fail_count += 1

# Print summary after processing all images
print("Done.")
print("Success:", success_count)
print("Failed:", fail_count)

Done.
Success: 149635
Failed: 4153
